In [2]:
from neo4j import GraphDatabase
import json
from pathlib import Path
from tqdm.auto import tqdm

PROJECT_ROOT  = Path.home() / "xai-knowledge-graph"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

def load_creds(path=str(Path.home() / "aura_cred.txt")):
    creds = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if "=" in line and not line.startswith("#"):
                k, v = line.split("=", 1)
                creds[k.strip()] = v.strip()
    return creds

creds  = load_creds()
driver = GraphDatabase.driver(
    creds["NEO4J_URI"],
    auth=(creds["NEO4J_USERNAME"], creds["NEO4J_PASSWORD"])
)
DB = creds["NEO4J_DATABASE"]

with driver.session(database=DB) as s:
    print(s.run("RETURN 'connected' AS status").single()["status"])


connected


In [3]:
with open(PROCESSED_DIR / "papers_final.json") as f:
    papers = json.load(f)

corpus_ids = {p["arxiv_id"] for p in papers}
print(f"Loaded {len(papers)} papers ({len(corpus_ids)} unique arxiv IDs)")


Loaded 3913 papers (3907 unique arxiv IDs)


In [4]:
with driver.session(database=DB) as s:
    s.run("MATCH (n) DETACH DELETE n")
    print("Database cleared")


Database cleared


In [5]:
constraints = [
    "CREATE CONSTRAINT paper_arxiv_id IF NOT EXISTS FOR (p:Paper)  REQUIRE p.arxiv_id IS UNIQUE",
    "CREATE CONSTRAINT author_name    IF NOT EXISTS FOR (a:Author) REQUIRE a.name     IS UNIQUE",
    "CREATE CONSTRAINT venue_name     IF NOT EXISTS FOR (v:Venue)  REQUIRE v.name     IS UNIQUE",
    "CREATE CONSTRAINT topic_name     IF NOT EXISTS FOR (t:Topic)  REQUIRE t.name     IS UNIQUE",
]
with driver.session(database=DB) as s:
    for c in constraints:
        s.run(c)
print("Constraints created")


Constraints created


In [6]:
PAPER_QUERY = """
UNWIND $batch AS p
MERGE (paper:Paper {arxiv_id: p.arxiv_id})
SET paper.title          = p.title,
    paper.abstract       = p.abstract,
    paper.year           = p.year,
    paper.published      = p.published,
    paper.url            = p.url,
    paper.doi            = p.doi,
    paper.s2_paper_id    = p.s2_paper_id,
    paper.citation_count = coalesce(p.citation_count, 0)
"""

BATCH = 500
with driver.session(database=DB) as s:
    for i in tqdm(range(0, len(papers), BATCH), desc="Papers"):
        s.run(PAPER_QUERY, batch=papers[i:i+BATCH])
print(f"Loaded {len(papers)} papers")


Papers:   0%|          | 0/8 [00:00<?, ?it/s]

Loaded 3913 papers


In [7]:
# Build deduplicated author list — prefer S2 authors (richer data) when available
author_data = {}
authored_edges = []

for p in papers:
    aid = p["arxiv_id"]
    if p.get("s2_authors"):
        for sa in p["s2_authors"]:
            if not sa.get("name"): continue
            name = sa["name"].strip()
            if name and name not in author_data:
                author_data[name] = {"name": name, "s2_author_id": sa.get("author_id")}
            if name:
                authored_edges.append({"author": name, "paper": aid})
    else:
        for name in p.get("authors", []):
            name = (name or "").strip()
            if name and name not in author_data:
                author_data[name] = {"name": name, "s2_author_id": None}
            if name:
                authored_edges.append({"author": name, "paper": aid})

print(f"Unique authors: {len(author_data):,}")
print(f"AUTHORED edges: {len(authored_edges):,}")

AUTHOR_QUERY = """
UNWIND $batch AS a
MERGE (author:Author {name: a.name})
SET author.s2_author_id = coalesce(a.s2_author_id, author.s2_author_id)
"""
AUTHORED_QUERY = """
UNWIND $batch AS pair
MATCH (a:Author {name: pair.author})
MATCH (p:Paper  {arxiv_id: pair.paper})
MERGE (a)-[:AUTHORED]->(p)
"""

author_list = list(author_data.values())
with driver.session(database=DB) as s:
    for i in tqdm(range(0, len(author_list), BATCH), desc="Authors"):
        s.run(AUTHOR_QUERY, batch=author_list[i:i+BATCH])
    for i in tqdm(range(0, len(authored_edges), BATCH), desc="AUTHORED"):
        s.run(AUTHORED_QUERY, batch=authored_edges[i:i+BATCH])
print(" Authors + AUTHORED loaded")

Unique authors: 13,933
AUTHORED edges: 17,150


Authors:   0%|          | 0/28 [00:00<?, ?it/s]

AUTHORED:   0%|          | 0/35 [00:00<?, ?it/s]

 Authors + AUTHORED loaded


In [8]:
venue_set    = set()
venue_edges  = []
for p in papers:
    v = (p.get("venue") or "").strip()
    if v:
        venue_set.add(v)
        venue_edges.append({"venue": v, "paper": p["arxiv_id"]})

print(f"Unique venues: {len(venue_set)} | PUBLISHED_IN edges: {len(venue_edges)}")

VENUE_QUERY    = "UNWIND $batch AS v MERGE (venue:Venue {name: v.name})"
PUBLISHED_Q    = """
UNWIND $batch AS pair
MATCH (v:Venue {name: pair.venue})
MATCH (p:Paper {arxiv_id: pair.paper})
MERGE (p)-[:PUBLISHED_IN]->(v)
"""

with driver.session(database=DB) as s:
    s.run(VENUE_QUERY, batch=[{"name": v} for v in venue_set])
    for i in tqdm(range(0, len(venue_edges), BATCH), desc="PUBLISHED_IN"):
        s.run(PUBLISHED_Q, batch=venue_edges[i:i+BATCH])
print(" Venues + PUBLISHED_IN loaded")

Unique venues: 733 | PUBLISHED_IN edges: 3305


PUBLISHED_IN:   0%|          | 0/7 [00:00<?, ?it/s]

 Venues + PUBLISHED_IN loaded


In [10]:
topic_set   = set()
topic_edges = []
for p in papers:
    for t in (p.get("topics") or []):
        topic_set.add(t)
        topic_edges.append({"topic": t, "paper": p["arxiv_id"]})

print(f"Unique topics: {len(topic_set)} | ABOUT edges: {len(topic_edges)}")

TOPIC_QUERY = "UNWIND $batch AS t MERGE (topic:Topic {name: t.name})"
ABOUT_Q     = """
UNWIND $batch AS pair
MATCH (t:Topic {name: pair.topic})
MATCH (p:Paper {arxiv_id: pair.paper})
MERGE (p)-[:ABOUT]->(t)
"""

with driver.session(database=DB) as s:
    s.run(TOPIC_QUERY, batch=[{"name": t} for t in topic_set])
    for i in tqdm(range(0, len(topic_edges), BATCH), desc="ABOUT"):
        s.run(ABOUT_Q, batch=topic_edges[i:i+BATCH])
print(" Topics + ABOUT loaded")

Unique topics: 30 | ABOUT edges: 11409


ABOUT:   0%|          | 0/23 [00:00<?, ?it/s]

 Topics + ABOUT loaded


In [11]:
cites_edges = []
for p in papers:
    citing = p["arxiv_id"]
    for ref in (p.get("references_arxiv_ids") or []):
        ref_norm = (ref or "").split("v")[0].strip()  # strip version suffix
        if ref_norm and ref_norm in corpus_ids and ref_norm != citing:
            cites_edges.append({"citing": citing, "cited": ref_norm})

print(f"In-corpus citation edges: {len(cites_edges):,}")
print(f"(External references excluded — keeps the citation graph internally consistent)")

CITES_Q = """
UNWIND $batch AS pair
MATCH (c:Paper {arxiv_id: pair.citing})
MATCH (t:Paper {arxiv_id: pair.cited})
MERGE (c)-[:CITES]->(t)
"""

with driver.session(database=DB) as s:
    for i in tqdm(range(0, len(cites_edges), BATCH), desc="CITES"):
        s.run(CITES_Q, batch=cites_edges[i:i+BATCH])
print(" CITES loaded")

In-corpus citation edges: 2,639
(External references excluded — keeps the citation graph internally consistent)


CITES:   0%|          | 0/6 [00:00<?, ?it/s]

 CITES loaded


In [3]:
queries = [
    ("Papers",            "MATCH (n:Paper)  RETURN count(n) AS c"),
    ("Authors",           "MATCH (n:Author) RETURN count(n) AS c"),
    ("Venues",            "MATCH (n:Venue)  RETURN count(n) AS c"),
    ("Topics",            "MATCH (n:Topic)  RETURN count(n) AS c"),
    ("AUTHORED edges",    "MATCH ()-[r:AUTHORED]->()    RETURN count(r) AS c"),
    ("CITES edges",       "MATCH ()-[r:CITES]->()       RETURN count(r) AS c"),
    ("PUBLISHED_IN edges","MATCH ()-[r:PUBLISHED_IN]->() RETURN count(r) AS c"),
    ("ABOUT edges",       "MATCH ()-[r:ABOUT]->()       RETURN count(r) AS c"),
]
print(f"{'Element':<22}{'Count':>10}")
print("-" * 32)
with driver.session(database=DB) as s:
    for label, q in queries:
        n = s.run(q).single()["c"]
        print(f"{label:<22}{n:>10,}")

Element                    Count
--------------------------------
Papers                     3,907
Authors                   13,933
Venues                       733
Topics                        30
AUTHORED edges            17,119
CITES edges                2,637
PUBLISHED_IN edges         3,299
ABOUT edges               11,396
